# TS-ResNet50 Teacher-Layer Ablation

This notebook compares the saved `TS-ResNet50` teacher-layer variants on the curated `x64` benchmark split.

It is designed to be artifact-first. By default it loads the main local `layer2` run plus the saved `layer1` reference summaries, then recreates comparison tables and plots without retraining. If you explicitly enable the rerun flag, it can also launch local teacher-layer variants from this repo.

## Submission Context

- Main reference run: `experiments/anomaly_detection/teacher_student/resnet50/x64/main/notebook.ipynb`
- Base config: `experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/train_config.toml`
- Layer-ablation artifact root: `experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/`
- Saved outputs written by this notebook:
  - CSV summaries in `artifacts/results/`
  - plots in `artifacts/plots/`
  - generated configs in `artifacts/generated_configs/` when local reruns are enabled

In [ ]:
from pathlib import Path
from copy import deepcopy
import json
import subprocess
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml

REPO_ROOT

## Paths And Notebook Flags

This cell declares the local artifact layout and the flags that control whether the notebook stays in analysis mode or launches new local runs.

Default behavior is lightweight:
- `RUN_VARIANT_SWEEP = False`
- `REUSE_EXISTING_LOCAL_RUNS = True`
- saved `layer2` and `layer1` reference results are loaded first

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/train_config.toml'
MAIN_RUN_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/main/artifacts/ts_resnet50'
REFERENCE_LAYER1_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/layer1_reference_runs'
GENERATED_CONFIG_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/generated_configs'
RESULTS_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/results'
PLOTS_DIR = REPO_ROOT / 'experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/plots'
LOCAL_SUMMARY_PATH = RESULTS_DIR / 'local_variant_summary.csv'
REFERENCE_SUMMARY_PATH = RESULTS_DIR / 'layer1_reference_summary.csv'
COMPARISON_SUMMARY_PATH = RESULTS_DIR / 'teacher_layer_comparison.csv'
BEST_BY_LAYER_PATH = RESULTS_DIR / 'best_by_teacher_layer.csv'

RUN_VARIANT_SWEEP = False
REUSE_EXISTING_LOCAL_RUNS = True
LOAD_MAIN_LAYER2_RESULTS = True
LOAD_REFERENCE_LAYER1_RESULTS = True
THRESHOLD_QUANTILE = 0.95

VARIANT_SPECS = [
    {
        'name': 'ts_resnet50_layer2_topk20_sw2p0_aw1p0',
        'teacher_layer': 'layer2',
        'topk_ratio': 0.20,
        'score_student_weight': 2.0,
        'score_autoencoder_weight': 1.0,
    },
    {
        'name': 'ts_resnet50_layer1_topk10_sw1p0_aw0p5',
        'teacher_layer': 'layer1',
        'topk_ratio': 0.10,
        'score_student_weight': 1.0,
        'score_autoencoder_weight': 0.5,
    },
    {
        'name': 'ts_resnet50_layer1_topk10_sw2p0_aw1p0',
        'teacher_layer': 'layer1',
        'topk_ratio': 0.10,
        'score_student_weight': 2.0,
        'score_autoencoder_weight': 1.0,
    },
    {
        'name': 'ts_resnet50_layer1_topk10_sw2p0_aw0p5',
        'teacher_layer': 'layer1',
        'topk_ratio': 0.10,
        'score_student_weight': 2.0,
        'score_autoencoder_weight': 0.5,
    },
    {
        'name': 'ts_resnet50_layer1_topk20_sw2p0_aw1p0',
        'teacher_layer': 'layer1',
        'topk_ratio': 0.20,
        'score_student_weight': 2.0,
        'score_autoencoder_weight': 1.0,
    },
    {
        'name': 'ts_resnet50_layer1_topk20_sw1p0_aw0p5',
        'teacher_layer': 'layer1',
        'topk_ratio': 0.20,
        'score_student_weight': 1.0,
        'score_autoencoder_weight': 0.5,
    },
]

for path in [GENERATED_CONFIG_DIR, RESULTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

base_config = load_toml(BASE_CONFIG_PATH)
pd.DataFrame(VARIANT_SPECS)

## Helper Functions

These helpers handle local reruns, normalize the saved snapshot CSVs, and convert everything into one common comparison table.

In [ ]:
def format_toml_value(value):
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(value)
    return json.dumps(str(value))


def write_simple_toml(path, config_dict):
    lines = []
    for section_name, section_values in config_dict.items():
        lines.append(f'[{section_name}]')
        for key, value in section_values.items():
            lines.append(f'{key} = {format_toml_value(value)}')
        lines.append('')
    path.write_text('\n'.join(lines).rstrip() + '\n', encoding='utf-8')


def stream_command(command, cwd):
    print('Running:', ' '.join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def build_variant_config(base_config, spec):
    config = deepcopy(base_config)
    config['run']['output_dir'] = f"experiments/anomaly_detection/teacher_student/resnet50/x64/layer_ablation/artifacts/{spec['name']}"
    config['model']['teacher_layer'] = spec['teacher_layer']
    config['model']['topk_ratio'] = float(spec['topk_ratio'])
    config['model']['score_student_weight'] = float(spec['score_student_weight'])
    config['model']['score_autoencoder_weight'] = float(spec['score_autoencoder_weight'])
    config['training']['resume_from'] = ''
    return config


def run_variant(spec, threshold_quantile, reuse_existing=True):
    variant_config = build_variant_config(base_config, spec)
    config_path = GENERATED_CONFIG_DIR / f"{spec['name']}.toml"
    write_simple_toml(config_path, variant_config)

    output_dir = REPO_ROOT / variant_config['run']['output_dir']
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / 'best_model.pt'
    evaluation_dir = output_dir / 'evaluation'
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    summary_path = evaluation_dir / 'summary.json'

    if reuse_existing and checkpoint_path.exists():
        print(f"Reusing checkpoint for {spec['name']}: {checkpoint_path}")
    else:
        stream_command(
            [
                sys.executable,
                '-u',
                REPO_ROOT / 'scripts' / 'train_ts_distillation.py',
                '--config',
                config_path,
            ],
            cwd=REPO_ROOT,
        )

    if reuse_existing and summary_path.exists():
        print(f"Reusing evaluation for {spec['name']}: {summary_path}")
    else:
        stream_command(
            [
                sys.executable,
                REPO_ROOT / 'scripts' / 'evaluate_reconstruction_model.py',
                '--checkpoint',
                checkpoint_path,
                '--config',
                config_path,
                '--model-type',
                'ts_distillation',
                '--threshold-quantile',
                str(threshold_quantile),
                '--output-dir',
                evaluation_dir,
            ],
            cwd=REPO_ROOT,
        )

    train_summary_path = output_dir / 'summary.json'
    if train_summary_path.exists():
        train_summary = json.loads(train_summary_path.read_text(encoding='utf-8'))
    else:
        train_summary = {}
    evaluation_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    metrics = evaluation_summary['metrics_at_validation_threshold']
    best_sweep = evaluation_summary['best_threshold_sweep']

    return {
        'variant': spec['name'],
        'source': 'local_rerun',
        'teacher_layer': spec['teacher_layer'],
        'topk_ratio': float(spec['topk_ratio']),
        'score_student_weight': float(spec['score_student_weight']),
        'score_autoencoder_weight': float(spec['score_autoencoder_weight']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'f1': float(metrics['f1']),
        'auroc': float(metrics['auroc']),
        'auprc': float(metrics['auprc']),
        'best_sweep_f1': float(best_sweep['f1']),
        'threshold': float(metrics['threshold']),
        'best_epoch': train_summary.get('best_epoch', None),
        'best_val_loss': train_summary.get('best_val_loss', None),
        'output_dir': str(output_dir),
        'config_path': str(config_path),
    }


def load_reference_layer1_rows(reference_dir):
    snapshot_paths = sorted(reference_dir.glob('leaderboard_snapshot_*.csv'))
    if not snapshot_paths:
        return pd.DataFrame()
    frames = []
    for snapshot_path in snapshot_paths:
        frame = pd.read_csv(snapshot_path).copy()
        frame['variant'] = frame['run_name']
        frame['source'] = 'saved_layer1_reference'
        frame['score_autoencoder_weight'] = frame['score_auto_weight']
        frame['output_dir'] = frame['run_dir']
        frame['config_path'] = snapshot_path.name
        frame['snapshot_name'] = snapshot_path.name
        frames.append(frame)
    reference_df = pd.concat(frames, ignore_index=True)
    reference_df = reference_df.sort_values(['variant', 'f1', 'best_sweep_f1', 'auroc'], ascending=[True, False, False, False])
    reference_df = reference_df.drop_duplicates(subset=['variant'], keep='first').reset_index(drop=True)
    keep_columns = [
        'variant', 'source', 'teacher_layer', 'topk_ratio', 'score_student_weight', 'score_autoencoder_weight',
        'precision', 'recall', 'f1', 'auroc', 'auprc', 'best_sweep_f1', 'threshold', 'best_epoch', 'best_val_loss',
        'output_dir', 'config_path', 'snapshot_name'
    ]
    return reference_df[keep_columns]


def load_main_layer2_rows(main_run_dir):
    evaluation_dir = main_run_dir / 'results' / 'evaluation'
    summary = json.loads((evaluation_dir / 'summary.json').read_text(encoding='utf-8'))
    selected = json.loads((evaluation_dir / 'selected_score_variant.json').read_text(encoding='utf-8'))
    rows = [
        {
            'variant': 'ts_resnet50_layer2_default_score',
            'source': 'main_layer2',
            'teacher_layer': 'layer2',
            'topk_ratio': 0.20,
            'score_student_weight': 1.0,
            'score_autoencoder_weight': 0.0,
            'precision': float(summary['metrics_at_validation_threshold']['precision']),
            'recall': float(summary['metrics_at_validation_threshold']['recall']),
            'f1': float(summary['metrics_at_validation_threshold']['f1']),
            'auroc': float(summary['metrics_at_validation_threshold']['auroc']),
            'auprc': float(summary['metrics_at_validation_threshold']['auprc']),
            'best_sweep_f1': float(summary['best_threshold_sweep']['f1']),
            'threshold': float(summary['metrics_at_validation_threshold']['threshold']),
            'best_epoch': None,
            'best_val_loss': None,
            'output_dir': str(main_run_dir),
            'config_path': str(BASE_CONFIG_PATH),
            'snapshot_name': 'main_layer2_default',
        },
        {
            'variant': 'ts_resnet50_layer2_selected_score',
            'source': 'main_layer2',
            'teacher_layer': 'layer2',
            'topk_ratio': float(selected['topk_ratio']),
            'score_student_weight': float(selected['student_weight']),
            'score_autoencoder_weight': float(selected['auto_weight']),
            'precision': float(selected['precision']),
            'recall': float(selected['recall']),
            'f1': float(selected['f1']),
            'auroc': float(selected['auroc']),
            'auprc': float(selected['auprc']),
            'best_sweep_f1': float(selected['best_sweep_f1']),
            'threshold': float(selected['threshold']),
            'best_epoch': None,
            'best_val_loss': None,
            'output_dir': str(main_run_dir),
            'config_path': str(BASE_CONFIG_PATH),
            'snapshot_name': 'main_layer2_selected',
        },
    ]
    return pd.DataFrame(rows)

## Load Saved Comparison Inputs

This cell loads the main `layer2` run and the saved `layer1` reference summaries, optionally adds local rerun results, and writes normalized comparison CSVs into `artifacts/results/`.

In [ ]:
main_df = load_main_layer2_rows(MAIN_RUN_DIR) if LOAD_MAIN_LAYER2_RESULTS else pd.DataFrame()
reference_df = load_reference_layer1_rows(REFERENCE_LAYER1_DIR) if LOAD_REFERENCE_LAYER1_RESULTS else pd.DataFrame()

if not reference_df.empty:
    reference_df.to_csv(REFERENCE_SUMMARY_PATH, index=False)

if RUN_VARIANT_SWEEP:
    local_rows = []
    for spec in VARIANT_SPECS:
        print(f"\n=== Variant: {spec['name']} ===")
        local_rows.append(run_variant(spec=spec, threshold_quantile=THRESHOLD_QUANTILE, reuse_existing=REUSE_EXISTING_LOCAL_RUNS))
    local_df = pd.DataFrame(local_rows).sort_values(['f1', 'auroc'], ascending=False).reset_index(drop=True)
    local_df.to_csv(LOCAL_SUMMARY_PATH, index=False)
elif LOCAL_SUMMARY_PATH.exists():
    local_df = pd.read_csv(LOCAL_SUMMARY_PATH)
    print(f'Loaded existing local summary from {LOCAL_SUMMARY_PATH}')
else:
    local_df = pd.DataFrame()
    print('Skipping local rerun. Set RUN_VARIANT_SWEEP = True to launch it.')

comparison_frames = [df for df in [main_df, reference_df, local_df] if not df.empty]
if comparison_frames:
    comparison_df = pd.concat(comparison_frames, ignore_index=True)
    comparison_df = comparison_df.sort_values(['f1', 'auroc', 'best_sweep_f1'], ascending=False).reset_index(drop=True)
else:
    comparison_df = pd.DataFrame()

if comparison_df.empty:
    raise RuntimeError('No comparison rows were loaded. Check the main run and layer1 reference artifacts first.')

comparison_df.to_csv(COMPARISON_SUMMARY_PATH, index=False)
best_by_layer_df = comparison_df.sort_values(['teacher_layer', 'f1', 'auroc'], ascending=[True, False, False]).groupby('teacher_layer', as_index=False).first()
best_by_layer_df.to_csv(BEST_BY_LAYER_PATH, index=False)

display(comparison_df[[
    'variant', 'source', 'teacher_layer', 'topk_ratio', 'score_student_weight', 'score_autoencoder_weight',
    'precision', 'recall', 'f1', 'auroc', 'auprc', 'best_sweep_f1'
]].round(6))

display(best_by_layer_df[[
    'variant', 'source', 'teacher_layer', 'topk_ratio', 'score_student_weight', 'score_autoencoder_weight',
    'precision', 'recall', 'f1', 'auroc', 'auprc', 'best_sweep_f1'
]].round(6))

## Save Comparison Plots

This cell recreates the main teacher-layer comparison figures and saves them into `artifacts/plots/`.

In [ ]:
plot_df = comparison_df.sort_values(['f1', 'auroc'], ascending=[False, False]).head(8).copy()
plot_df = plot_df.iloc[::-1].reset_index(drop=True)
colors = plot_df['teacher_layer'].map({'layer1': '#e76f51', 'layer2': '#2a9d8f'}).fillna('#8d99ae')

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(plot_df['variant'], plot_df['f1'], color=colors)
ax.set_title('Top Teacher-Layer Variants By F1')
ax.set_xlabel('f1')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'top_variants_f1.png', dpi=200, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5.5))
layer_colors = comparison_df['teacher_layer'].map({'layer1': '#e76f51', 'layer2': '#2a9d8f'}).fillna('#8d99ae')
ax.scatter(comparison_df['recall'], comparison_df['precision'], c=layer_colors, s=75, alpha=0.85)
for _, row in comparison_df.head(6).iterrows():
    ax.annotate(row['variant'], (row['recall'], row['precision']), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_title('Precision vs Recall Across Teacher-Layer Variants')
ax.set_xlabel('recall')
ax.set_ylabel('precision')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'precision_recall_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(best_by_layer_df['teacher_layer'], best_by_layer_df['f1'], color=['#e76f51' if layer == 'layer1' else '#2a9d8f' for layer in best_by_layer_df['teacher_layer']])
axes[0].set_title('Best F1 By Teacher Layer')
axes[0].set_ylabel('f1')
axes[1].bar(best_by_layer_df['teacher_layer'], best_by_layer_df['auroc'], color=['#e76f51' if layer == 'layer1' else '#2a9d8f' for layer in best_by_layer_df['teacher_layer']])
axes[1].set_title('Best AUROC By Teacher Layer')
axes[1].set_ylabel('auroc')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'best_by_teacher_layer.png', dpi=200, bbox_inches='tight')
plt.show()

## Notes

- `layer2` remains the strongest saved ResNet50 teacher-student direction in the curated local benchmark.
- `layer1` still matters because it gives us a direct teacher-layer comparison instead of assuming the deeper layer is always best.
- Leaving `RUN_VARIANT_SWEEP = False` keeps this notebook fast and submission-friendly.
- Turning the flag on makes the notebook a real local rerun workflow instead of only a summary notebook.